# Live Mistral PDF Smoke Test

This notebook tests the thing that matters: **can Mistral OCR process a Gazette PDF URL through this package and produce usable bundle files?**

It makes a live Mistral API call against a public Kenya Gazette PDF URL, caches the raw Mistral OCR JSON, runs the package parser, validates the envelope, writes bundles, and previews the output.

You need `MISTRAL_API_KEY` in the notebook kernel environment or in the repo `.env` file. This is a real network/API test and may be billable.


## What You Are Testing

Run the cells top to bottom. A successful run proves:

1. The notebook kernel can import this package.
2. The kernel has access to `MISTRAL_API_KEY`.
3. Mistral accepts the configured public PDF URL and returns OCR JSON.
4. The package caches that raw OCR JSON under `examples/_live_mistral_pdf_test/stage`.
5. The package converts OCR markdown into an envelope and validates it.
6. The bundle writer creates JSON and markdown outputs under `examples/_live_mistral_pdf_test/bundles`.

The package also supports live local PDF paths through `parse_file(...)`: it uploads the file to Mistral first, then OCRs the uploaded `file_id`. The main notebook path still uses a public URL because it is reproducible on any machine.

In [9]:
from pathlib import Path
import os
import sys

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

cwd = Path.cwd()
repo_root = next(
    (
        candidate
        for candidate in (cwd, cwd.parent)
        if (candidate / "gazette_mistral_pipeline" / "__init__.py").is_file()
    ),
    None,
)
if repo_root is not None and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from gazette_mistral_pipeline import (
        Bundles,
        GazetteConfig,
        parse_file,
        parse_url,
        validate_envelope_json,
        write_envelope,
    )
except ModuleNotFoundError as exc:
    if exc.name == "pydantic" and repo_root is not None:
        raise ModuleNotFoundError(
            "This notebook kernel is missing package dependencies. "
            "Run this in a notebook cell, then restart the kernel: "
            f'%pip install -e "{repo_root}"'
        ) from exc
    raise

base_dir = repo_root or cwd


def show_markdown(text: str) -> None:
    # Always print plain text so Cursor/Jupyter cannot hide the important result.
    print(text)
    if Markdown is not None and display is not None:
        display(Markdown(text))


def display_path(path: Path) -> str:
    try:
        return str(Path(path).resolve().relative_to(base_dir.resolve())).replace("\\", "/")
    except ValueError:
        return str(path)


def load_env_file(path: Path) -> bool:
    if not path.is_file():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
    return True


dotenv_loaded = load_env_file(base_dir / ".env")
api_key_ready = bool(os.environ.get("MISTRAL_API_KEY", "").strip())

show_markdown(
    "## Setup Ready\n\n"
    f"- Package import: `ok`\n"
    f"- Repo root: `{display_path(base_dir)}`\n"
    f"- `.env` loaded: `{dotenv_loaded}`\n"
    f"- `MISTRAL_API_KEY` available: `{api_key_ready}`"
)


## Setup Ready

- Package import: `ok`
- Repo root: `.`
- `.env` loaded: `True`
- `MISTRAL_API_KEY` available: `True`


## Setup Ready

- Package import: `ok`
- Repo root: `.`
- `.env` loaded: `True`
- `MISTRAL_API_KEY` available: `True`

## Live PDF Configuration

This cell chooses the public PDF URL and output folders for the live Mistral run. Change `PDF_URL` if you want to test a different public PDF URL.


In [ ]:
PDF_URL = "https://new.kenyalaw.org/akn/ke/officialGazette/2009-12-11/103/eng@2009-12-11/source.pdf"

examples_dir = base_dir / "examples"
output_root = examples_dir / "_live_mistral_pdf_test"
stage_dir = output_root / "stage"
bundle_dir = output_root / "bundles"

live_config = GazetteConfig(
    runtime={
        "allow_live_mistral": True,
        "output_dir": stage_dir,
    }
)

show_markdown(
    "## Live Test Configuration\n\n"
    f"- PDF URL: `{PDF_URL}`\n"
    f"- Mistral model: `{live_config.mistral.model}`\n"
    f"- API key env var: `{live_config.mistral.api_key_env}`\n"
    f"- Stage directory: `{display_path(stage_dir)}`\n"
    f"- Bundle directory: `{display_path(bundle_dir)}`"
)


## Run Live Mistral OCR On The PDF

This is the real test. It calls Mistral OCR using `document_url`, caches the raw OCR JSON, validates the parsed envelope, and writes the selected bundles.

If this cell fails with a missing API key error, set `MISTRAL_API_KEY` in `.env` or in the notebook kernel environment and rerun from the setup cell.


In [ ]:
if not os.environ.get("MISTRAL_API_KEY", "").strip():
    raise RuntimeError(
        "Missing MISTRAL_API_KEY. Add it to .env or the notebook kernel environment, "
        "then rerun the setup cell."
    )

live_env = parse_url(PDF_URL, config=live_config)

written = write_envelope(
    live_env,
    bundle_dir,
    Bundles(notices=True, tables=True, document_index=True, schema=True),
)

validated = validate_envelope_json(written["envelope"])

summary = {
    "run_name": validated.source.run_name,
    "mistral_model": validated.mistral.model,
    "mistral_replay": validated.mistral.request_options.get("replay"),
    "page_count": validated.stats.page_count,
    "notice_count": validated.stats.notice_count,
    "table_count": validated.stats.table_count,
    "raw_json_path": display_path(Path(validated.mistral.raw_json_path)),
    "written_bundles": sorted(written),
}

bundle_lines = "\n".join(
    f"- `{bundle_name}`: `{display_path(bundle_path)}`"
    for bundle_name, bundle_path in sorted(written.items())
)
report = f"""## Live Mistral PDF Test Complete

- Source URL: `{PDF_URL}`
- Run name: `{summary['run_name']}`
- Mistral model: `{summary['mistral_model']}`
- Replay mode: `{summary['mistral_replay']}`
- Page count: `{summary['page_count']}`
- Notice count: `{summary['notice_count']}`
- Table count: `{summary['table_count']}`
- Raw Mistral JSON cache: `{summary['raw_json_path']}`

Written bundles:
{bundle_lines}
"""

show_markdown(report)
summary


In [ ]:
import json

raw_payload = json.loads(Path(validated.mistral.raw_json_path).read_text(encoding="utf-8"))
joined_preview = Path(written["joined_markdown"]).read_text(encoding="utf-8").strip()
notices_payload = json.loads(Path(written["notices"]).read_text(encoding="utf-8"))
first_notice = notices_payload[0] if notices_payload else None
first_table = first_notice["tables"][0] if first_notice and first_notice["tables"] else None
joined_excerpt = joined_preview[:2500]

if first_notice is None:
    notice_summary = "No notices were parsed from this OCR response."
else:
    notice_summary = (
        f"- Notice ID: `{first_notice['notice_id']}`\n"
        f"- Notice number: `{first_notice.get('notice_no')}`\n"
        f"- Title lines: `{'; '.join(first_notice.get('title_lines', []))}`\n"
        f"- Dates found: `{', '.join(first_notice.get('dates_found', []))}`\n"
        f"- Confidence band: `{first_notice['confidence_scores']['band']}`"
    )

if first_table is None:
    table_summary = "No tables were parsed from the first notice."
else:
    table_summary = (
        f"- Headers: `{', '.join(first_table['headers'])}`\n"
        f"- First row: `{first_table['records'][0] if first_table['records'] else {}}`"
    )

preview = f"""## Live OCR Output Preview

Raw Mistral cache file: `{display_path(Path(validated.mistral.raw_json_path))}`

Raw Mistral top-level keys: `{', '.join(sorted(raw_payload.keys()))}`

First parsed notice:

{notice_summary}

First parsed table:

{table_summary}

Joined markdown excerpt:

```text
{joined_excerpt}
```
"""

show_markdown(preview)

## How To Judge The Test

The live test worked if the run cell shows:

- `Live Mistral PDF Test Complete`
- `Replay mode: False`
- a raw JSON cache under `examples/_live_mistral_pdf_test/stage`
- bundle files under `examples/_live_mistral_pdf_test/bundles`
- a joined markdown excerpt from the PDF

If Mistral returns OCR but the parser finds few or zero notices, that is a parser/data-quality issue, not a Mistral connectivity failure.

## Optional: Change The PDF URL

To test another public PDF URL, change `PDF_URL` in the configuration cell and rerun the configuration, live Mistral, and preview cells.

For private local files, use the `parse_file(...)` pattern in the next section instead of `parse_url(...)`.


In [ ]:
show_markdown(
    "## Current PDF URL\n\n"
    f"`{PDF_URL}`\n\n"
    "Edit `PDF_URL` in the configuration cell above to test a different public PDF."
)


## Local PDF Files

You can run this section after the setup cell. It does **not** require the public URL parser cell to run first.

Local or network PDF paths now work through `parse_file(...)` when live Mistral mode is enabled. The package reads the PDF from your machine, uploads it to Mistral Files with `purpose="ocr"`, then sends the returned `file_id` to the OCR endpoint.

Use this for paths such as `C:\\path\\to\\Gazette.pdf` or `\\\\server\\share\\Gazette.pdf`. The file must be readable by your Python process; Mistral does not need direct access to your local path.


In [ ]:
# Keep this cell runnable if the notebook kernel still has older setup-cell imports loaded.
if "parse_file" not in globals():
    from gazette_mistral_pipeline import parse_file

LOCAL_PDF = base_dir / "pdfs" / "Kenya Gazette Vol CXXVIINo 258.pdf"

if LOCAL_PDF is None:
    show_markdown(
        "## Local PDF Example\n\n"
        "Set `LOCAL_PDF` to a real `.pdf` path and rerun this cell to test local live OCR. "
        "The package will upload the PDF to Mistral, OCR the returned `file_id`, and write bundles under the local output folder."
    )
elif not Path(LOCAL_PDF).is_file():
    raise FileNotFoundError(f"Local PDF does not exist: {LOCAL_PDF}")
elif not os.environ.get("MISTRAL_API_KEY", "").strip():
    raise RuntimeError(
        "Missing MISTRAL_API_KEY. Add it to .env or the notebook kernel environment, "
        "then rerun the setup cell."
    )
else:
    local_examples_dir = base_dir / "examples"
    local_output_root = local_examples_dir / "_live_local_pdf_test"
    local_stage_dir = local_output_root / "stage"
    local_bundle_dir = local_output_root / "bundles"
    local_config = GazetteConfig(
        runtime={
            "allow_live_mistral": True,
            "output_dir": local_stage_dir,
        }
    )
    local_env = parse_file(Path(LOCAL_PDF), config=local_config)
    local_written = write_envelope(
        local_env,
        local_bundle_dir,
        Bundles(notices=True, tables=True, document_index=True, schema=True),
    )
    show_markdown(
        "## Local PDF OCR Complete\n\n"
        f"- Source path: `{display_path(Path(LOCAL_PDF))}`\n"
        f"- Run name: `{local_env.source.run_name}`\n"
        f"- Page count: `{local_env.stats.page_count}`\n"
        f"- Notice count: `{local_env.stats.notice_count}`\n"
        f"- Raw Mistral JSON cache: `{display_path(Path(local_env.mistral.raw_json_path))}`\n"
        f"- Envelope: `{display_path(local_written['envelope'])}`"
    )
